# Topic 2: Airflow + Spark — AWS Operators

> **Phase 7 → Week 14 → Topic 2**

---

## Interview Cheat Sheet

**Q: How do you orchestrate an EMR Spark job from Airflow?**
> Four-step pattern: (1) `EmrCreateJobFlowOperator` creates the cluster, (2) `EmrAddStepsOperator` submits the Spark step, (3) `EmrStepSensor` waits for the step to complete, (4) `EmrTerminateJobFlowOperator` terminates the cluster. The cluster ID is passed between tasks via XCom. Always use `catchup=False` and a `terminate_cluster` task with `trigger_rule='all_done'` so the cluster is cleaned up even on failure.

**Q: What is SparkSubmitOperator and when would you use it over EMR operators?**
> `SparkSubmitOperator` runs `spark-submit` on the Airflow worker or a connected Spark cluster (standalone or YARN). Use it when: Airflow is co-located with the Spark cluster, or for local testing. For production AWS workloads, prefer EMR operators — they create ephemeral clusters (pay only while running), are fully managed, and integrate with IAM/S3 natively.

**Q: What is the GlueJobOperator and when would you choose Glue over EMR for orchestration?**
> `GlueJobOperator` starts a Glue ETL job and (optionally) waits for completion. Choose Glue when: (1) job is simple S3→S3 ETL without complex Spark tuning, (2) team wants zero cluster management, (3) job runs infrequently. Choose EMR when: (4) streaming, (5) custom packages, (6) cost-sensitive (Spot instances), (7) need Spark UI for debugging.

In [ ]:
print("Airflow AWS Operators — reference patterns")
print("Requires: pip install apache-airflow-providers-amazon")

## Part 1: EMR Operator Pattern

In [ ]:
print("""
Full EMR Lifecycle in Airflow:
══════════════════════════════════════════════════════════════

from airflow import DAG
from airflow.providers.amazon.aws.operators.emr import (
    EmrCreateJobFlowOperator,
    EmrAddStepsOperator,
    EmrTerminateJobFlowOperator,
)
from airflow.providers.amazon.aws.sensors.emr import EmrStepSensor
from airflow.operators.dummy import DummyOperator
from datetime import datetime, timedelta

CLUSTER_CONFIG = {
    'Name': 'Daily ETL {{ ds }}',
    'ReleaseLabel': 'emr-6.15.0',
    'Applications': [{'Name': 'Spark'}],
    'Instances': {
        'InstanceGroups': [
            {'Name': 'Master', 'InstanceRole': 'MASTER',
             'InstanceType': 'm5.xlarge', 'InstanceCount': 1},
            {'Name': 'Core',   'InstanceRole': 'CORE',
             'InstanceType': 'm5.2xlarge', 'InstanceCount': 2},
            {'Name': 'Task',   'InstanceRole': 'TASK',
             'InstanceType': 'm5.2xlarge', 'InstanceCount': 6,
             'BidPrice': 'OnDemandPrice'},  # Spot
        ],
        'KeepJobFlowAliveWhenNoSteps': True,  # don't auto-terminate
        'TerminationProtected': False,
    },
    'JobFlowRole': 'EMR_EC2_DefaultRole',
    'ServiceRole': 'EMR_DefaultRole',
    'LogUri': 's3://my-logs/emr/{{ ds }}/',
    'BootstrapActions': [{
        'Name': 'Install deps',
        'ScriptBootstrapAction': {'Path': 's3://my-bucket/bootstrap.sh'}
    }],
    'Configurations': [{
        'Classification': 'spark-defaults',
        'Properties': {
            'spark.sql.adaptive.enabled': 'true',
            'spark.executor.memory': '18g',
            'spark.executor.cores': '5',
        }
    }]
}

SPARK_STEPS = [
    {
        'Name': 'Bronze Ingest',
        'ActionOnFailure': 'CONTINUE',
        'HadoopJarStep': {
            'Jar': 'command-runner.jar',
            'Args': [
                'spark-submit', '--deploy-mode', 'cluster',
                '--py-files', 's3://my-bucket/libs.zip',
                's3://my-bucket/scripts/bronze_ingest.py',
                '--date', '{{ ds }}'
            ]
        }
    },
    {
        'Name': 'Silver Transform',
        'ActionOnFailure': 'CONTINUE',
        'HadoopJarStep': {
            'Jar': 'command-runner.jar',
            'Args': ['spark-submit', '--deploy-mode', 'cluster',
                     's3://my-bucket/scripts/silver_transform.py',
                     '--date', '{{ ds }}']
        }
    },
]

with DAG('daily_emr_pipeline', schedule_interval='@daily',
         start_date=datetime(2024,1,1), catchup=False) as dag:

    create_cluster = EmrCreateJobFlowOperator(
        task_id='create_emr_cluster',
        job_flow_overrides=CLUSTER_CONFIG,
        aws_conn_id='aws_default',
        # Pushes cluster ID to XCom automatically
    )

    add_steps = EmrAddStepsOperator(
        task_id='add_spark_steps',
        job_flow_id="{{ task_instance.xcom_pull('create_emr_cluster', key='return_value') }}",
        aws_conn_id='aws_default',
        steps=SPARK_STEPS,
        # Returns list of step IDs
    )

    wait_for_steps = EmrStepSensor(
        task_id='wait_for_steps',
        job_flow_id="{{ task_instance.xcom_pull('create_emr_cluster', key='return_value') }}",
        step_id="{{ task_instance.xcom_pull('add_spark_steps', key='return_value')[0] }}",
        aws_conn_id='aws_default',
        poke_interval=60,
        mode='reschedule',
    )

    # Always terminate — even on failure
    terminate_cluster = EmrTerminateJobFlowOperator(
        task_id='terminate_cluster',
        job_flow_id="{{ task_instance.xcom_pull('create_emr_cluster', key='return_value') }}",
        aws_conn_id='aws_default',
        trigger_rule='all_done',   # ← runs even if upstream tasks failed
    )

    create_cluster >> add_steps >> wait_for_steps >> terminate_cluster
══════════════════════════════════════════════════════════════
""")

## Part 2: Glue Job Operator

In [ ]:
print("""
Glue Job Operator Pattern:
══════════════════════════════════════════════════════════════

from airflow.providers.amazon.aws.operators.glue import GlueJobOperator
from airflow.providers.amazon.aws.sensors.glue import GlueJobSensor

with DAG('daily_glue_pipeline', schedule_interval='@daily', ...) as dag:

    # Option A: run_job_flow=True → waits for completion (simpler)
    run_glue_job = GlueJobOperator(
        task_id='run_glue_etl',
        job_name='my-orders-etl',
        script_location='s3://my-bucket/scripts/etl.py',
        s3_bucket='my-bucket',
        iam_role_name='GlueRole',
        create_job_kwargs={
            'GlueVersion': '4.0',
            'NumberOfWorkers': 10,
            'WorkerType': 'G.1X',
        },
        script_args={
            '--date': '{{ ds }}',
            '--input_path': 's3://my-bucket/raw/{{ ds }}/',
            '--output_path': 's3://my-bucket/silver/{{ ds }}/',
            '--job-bookmark-option': 'job-bookmark-enable',
        },
        run_job_flow=True,   # wait for completion (blocks task)
        aws_conn_id='aws_default',
    )

    # Option B: separate operator + sensor (non-blocking)
    trigger_glue = GlueJobOperator(
        task_id='trigger_glue',
        job_name='my-orders-etl',
        run_job_flow=False,   # just trigger, don't wait
        ...
    )

    wait_for_glue = GlueJobSensor(
        task_id='wait_for_glue',
        job_name='my-orders-etl',
        run_id="{{ task_instance.xcom_pull('trigger_glue') }}",
        poke_interval=60,
        mode='reschedule',
        aws_conn_id='aws_default',
    )

    trigger_glue >> wait_for_glue

Glue Worker Types:
  Standard:  2 vCPU, 8 GB, 50 GB disk
  G.1X:      4 vCPU, 16 GB, 64 GB disk  ← default for most jobs
  G.2X:      8 vCPU, 32 GB, 128 GB disk  ← for memory-heavy jobs
  G.025X:    2 vCPU, 4 GB, 64 GB disk   ← cheapest, for small jobs

DPU = Data Processing Unit = 4 vCPU + 16 GB = 1 G.1X worker
Cost: G.1X = $0.44/DPU-hour per worker
══════════════════════════════════════════════════════════════
""")

## Part 3: SparkSubmitOperator (local/YARN)

In [ ]:
print("""
SparkSubmitOperator — Local or YARN Spark:
══════════════════════════════════════════════════════════════

from airflow.providers.apache.spark.operators.spark_submit import SparkSubmitOperator

# Connection: set up 'spark_default' connection in Airflow UI
# Type: Spark, Host: spark://spark-master:7077 (standalone) or yarn

submit_job = SparkSubmitOperator(
    task_id='run_spark_job',
    application='s3://my-bucket/scripts/silver_transform.py',  # or local path
    conn_id='spark_default',
    name='Silver Transform {{ ds }}',
    conf={
        'spark.executor.memory': '8g',
        'spark.executor.cores': '4',
        'spark.sql.shuffle.partitions': '200',
        'spark.sql.adaptive.enabled': 'true',
    },
    application_args=['--date', '{{ ds }}', '--env', 'prod'],
    py_files='s3://my-bucket/libs.zip',
    packages='io.delta:delta-spark_2.12:3.0.0',
    deploy_mode='cluster',   # or 'client'
    verbose=True,
)

SparkSubmitOperator vs EMR operators:
  SparkSubmitOperator:
    + Simple: runs spark-submit from the Airflow worker
    + Good for: Airflow running on YARN cluster, local testing
    - Requires persistent Spark cluster (paying for idle time)
    - Driver runs on Airflow worker (client mode) or remotely (cluster)
    - Hard to manage cluster lifecycle from Airflow

  EMR operators:
    + Ephemeral clusters (pay only for job duration)
    + Full AWS integration (IAM, S3, CloudWatch)
    + Easy Spot instance configuration
    - More setup (cluster config JSON)
    - Only for AWS environments

Local Docker testing with SparkSubmitOperator:
  # In your docker-compose.yml, add Airflow service connected to Spark network
  # SparkSubmitOperator can submit to spark://spark-master:7077
  # This is how our learn_spark cluster works!
══════════════════════════════════════════════════════════════
""")

## Part 4: Connections & Variables

In [ ]:
print("""
Airflow Connections & Variables:
══════════════════════════════════════════════════════════════

Connections: store external system credentials (encrypted in Airflow DB)
  Admin → Connections → Add
  Or via CLI:
    airflow connections add 'aws_default' \\
      --conn-type 'aws' \\
      --conn-extra '{"aws_access_key_id": "AK...", "aws_secret_access_key": "...", "region_name": "us-east-1"}'

  Or environment variables (preferred for security):
    AIRFLOW_CONN_AWS_DEFAULT=aws://AK...:secret@?region_name=us-east-1
    AIRFLOW_CONN_SPARK_DEFAULT=spark://spark-master:7077

Variables: store configuration values (not credentials)
  Admin → Variables → Add
  Or CLI:
    airflow variables set 'emr_cluster_id' 'j-XXXX'
    airflow variables set 'data_bucket' 'my-data-bucket'

  Usage in Python:
    from airflow.models import Variable
    bucket = Variable.get('data_bucket')           # plain string
    config = Variable.get('job_config', deserialize_json=True)  # JSON → dict

  Usage in Jinja:
    bash_command='aws s3 ls s3://{{ var.value.data_bucket }}/{{ ds }}/'

Secrets backend (production):
  Store credentials in AWS Secrets Manager, HashiCorp Vault, or GCP Secret Manager
  Configure Airflow to look up secrets automatically:

  # airflow.cfg:
  [secrets]
  backend = airflow.providers.amazon.aws.secrets.secrets_manager.SecretsManagerBackend
  backend_kwargs = {"connections_prefix": "airflow/connections",
                    "variables_prefix": "airflow/variables"}

  AWS Secrets Manager secret name: airflow/connections/aws_default
  Airflow auto-resolves connection when operators reference 'aws_default'
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Write a complete Airflow DAG that creates an EMR cluster, submits three sequential Spark steps (Bronze → Silver → Gold), waits for each step, and terminates the cluster even if any step fails. Pass the cluster ID between tasks using XCom.
2. Modify the DAG from exercise 1 to: (a) wait for an S3 sensor before creating the cluster, (b) send a Slack notification on failure, (c) add an SLA of 4 hours on the Gold step.
3. Compare the total Airflow task execution time for: (a) Sensor in `poke` mode checking every 30 seconds for 30 minutes with 10 concurrent DAG runs, (b) Sensor in `reschedule` mode. Which exhausts the Airflow worker pool first?
4. Write a GlueJobOperator task that passes a date parameter and enables job bookmarks. How would you add a downstream task that validates the output row count using an Athena query?
5. What happens if `terminate_cluster` uses `trigger_rule='all_success'` instead of `'all_done'`? Give a scenario where this causes a running EMR cluster to never terminate.